# Ejercicios del Día 3 · El sistema por dentro

### Sesiones 7, 8 y 9

Estos ejercicios son para que practiques por tu cuenta lo que acabamos de ver en clase. No son
un examen: puedes equivocarte todas las veces que quieras y volver a ejecutar.

Cada ejercicio funciona igual. Hay una celda donde tú escribes algo, y debajo una celda de
comprobación que te dice si va bien y, si no, qué revisar. Ejecuta las dos.

Antes de empezar, conviene que hayas ejecutado `01_RAG_basico.ipynb` y `03_Buscar_y_responder.ipynb`.

Tiempo estimado: 35 minutos.

In [ ]:
# Esta celda prepara la comprobación de los ejercicios. Ejecútala primero.

def comprobar(numero, condicion, bien, mal):
    """Revisa una respuesta y explica el resultado."""
    marca = "CORRECTO" if condicion else "REVISAR"
    print(f"[{marca}] Ejercicio {numero}")
    print(f"   {bien if condicion else mal}")


print("Listo. Ya puedes resolver los ejercicios.")

Estos ejercicios usan el corpus de TiendaSol. La primera celda lo monta; ejecútala y espera a que
termine antes de seguir.

In [ ]:
import re, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

from langchain_community.vectorstores import LanceDB
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

MODELO_EMBEDDINGS = "embeddinggemma:300m"

documentos = []
for ruta in sorted(Path("corpus/corpus_tiendasol").glob("*.md")):
    texto = ruta.read_text(encoding="utf-8")
    documentos.append(Document(page_content=texto,
                               metadata={"doc_id": re.search(r"doc_id:\s*(\S+)", texto).group(1)}))

divisor = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
fragmentos = divisor.split_documents(documentos)
embeddings = OllamaEmbeddings(model=MODELO_EMBEDDINGS)
almacen = LanceDB.from_documents(fragmentos, embeddings, uri="/tmp/lancedb_ej3",
                                 table_name="ej", mode="overwrite")

print(f"{len(documentos)} documentos -> {len(fragmentos)} fragmentos. Listo.")

## Ejercicio 3.1 · El tamaño del fragmento cambia el resultado

Vuelve a fragmentar el mismo corpus, pero con fragmentos de 200 caracteres y sin traslape.
Compara cuántos fragmentos salen.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ
# Pista: crea otro RecursiveCharacterTextSplitter con chunk_size=200 y chunk_overlap=0,
# y aplícalo a la misma lista "documentos".

fragmentos_chicos = None

In [ ]:
comprobar("3.1",
          fragmentos_chicos is not None and len(fragmentos_chicos) > len(fragmentos) * 2,
          f"Con fragmentos de 200 salen {len(fragmentos_chicos)}, contra "
          f"{len(fragmentos)} con los de 500. Más del doble.",
          "Deberían salir bastantes más fragmentos que con 500 caracteres.")

## Ejercicio 3.2 · Encontrar una pregunta que falle

En clase vimos que "¿en cuánto tiempo me regresan mi plata?" no recupera el documento de
reembolsos, porque la palabra "plata" no está en el corpus.

Encuentra tú otra pregunta que falle. Tiene que ser una pregunta razonable de un cliente, cuya
respuesta **sí esté** en el corpus, pero que recupere el documento equivocado.

Aviso: es más difícil de lo que parece. Probando diez preguntas coloquiales, solo una falló. Si no
encuentras ninguna después de intentarlo en serio, eso **también es un resultado**: significa que
el recuperador aguanta bien las variaciones de lenguaje en este corpus. Anótalo y escribe abajo
cuántas probaste.

In [ ]:
mi_pregunta = "ESCRIBE AQUÍ TU PREGUNTA"
documento_que_esperabas = "DOC-XXX-01"   # el doc_id donde está la respuesta

resultados = almacen.similarity_search(mi_pregunta, k=3)
recuperados = [d.metadata["doc_id"] for d in resultados]

print(f"pregunta : {mi_pregunta}")
print(f"esperabas: {documento_que_esperabas}")
print(f"recuperó : {recuperados}\n")
for d in resultados:
    print(f"  {d.metadata['doc_id']}: {' '.join(d.page_content.split())[:110]}...")

In [ ]:
escribiste = mi_pregunta != "ESCRIBE AQUÍ TU PREGUNTA"
fallo = escribiste and documento_que_esperabas not in recuperados

if not escribiste:
    print("[REVISAR] Ejercicio 3.2")
    print("   Escribe una pregunta y el documento donde debería estar la respuesta.")
elif fallo:
    print("[HALLAZGO] Ejercicio 3.2")
    print("   Encontraste una pregunta que rompe el sistema. Anótala y tráela a clase:")
    print("   casos así son exactamente los que alimentan un gold set de verdad.")
else:
    print("[CORRECTO] Ejercicio 3.2")
    print("   Tu pregunta sí recuperó el documento correcto. Sigue intentando con otras,")
    print("   y si tras varias no logras romperlo, esa es tu conclusión: el recuperador")
    print("   aguanta bien el lenguaje coloquial en este corpus. Anota cuántas probaste.")

## Ejercicio 3.3 · Cuántos fragmentos pedir

Con la pregunta "¿cuánto cuesta el envío y en cuánto tiempo llega?", que necesita dos documentos,
averigua cuál es el valor de k más pequeño que recupera **los dos** documentos correctos:
`DOC-COS-01` y `DOC-ENV-01`.

In [ ]:
PREGUNTA = "¿Cuánto cuesta el envío y en cuánto tiempo llega?"
NECESARIOS = {"DOC-COS-01", "DOC-ENV-01"}

# ESCRIBE TU CÓDIGO AQUÍ
# Pista: prueba con k = 1, 2, 3... y mira cuándo aparecen los dos documentos.

k_minimo = None

In [ ]:
res = almacen.similarity_search(PREGUNTA, k=k_minimo) if k_minimo else []
obtenidos = {d.metadata["doc_id"] for d in res}
menor = almacen.similarity_search(PREGUNTA, k=k_minimo - 1) if k_minimo and k_minimo > 1 else []
obtenidos_menor = {d.metadata["doc_id"] for d in menor}

comprobar("3.3",
          k_minimo is not None and NECESARIOS <= obtenidos and not NECESARIOS <= obtenidos_menor,
          f"Correcto: con k={k_minimo} aparecen los dos, y con k={k_minimo-1} todavía no.",
          "Busca el k más pequeño que traiga DOC-COS-01 y DOC-ENV-01 a la vez.")

## Para cerrar

Si algún ejercicio te quedó marcado como REVISAR y no encuentras por qué, anótalo y pregúntalo
mañana al empezar. Es información útil: si a varios les pasó lo mismo, conviene revisarlo con
todo el grupo.

Y si terminaste antes de tiempo, la mejor forma de aprovechar el rato es volver al cuaderno del
día y cambiar cosas a propósito para ver qué se rompe. Aprender qué rompe un sistema enseña más
que verlo funcionar.